# Atividade: Assistente de IA Generativa com Hugging Face e Gemini

**Disciplina:** Disruptive Architectures: IoT, IoB & Generative AI

**Grupo:**
- Vinicius Soteras Braga (RM566230)

**Tema escolhido:** Consultor de eficiência energética

---

Neste notebook o grupo vai construir um assistente de IA em 4 etapas (e 1 bônus):

1. Assistente com personalidade (Hugging Face)
2. Comparação Hugging Face x Gemini
3. Chat com memória
4. Interface web com Gradio
5. (Bônus) API com FastAPI

Os trechos marcados com **`# >>> PERSONALIZE`** devem ser alterados pelo grupo.
Execute as células em ordem, de cima para baixo.

## 0. Configuração

In [1]:
# huggingface_hub -> cliente para chamar modelos remotamente (API do Hugging Face)
# google-genai    -> SDK oficial do Google Gemini
# gradio          -> cria interfaces web a partir de funções Python
!pip install huggingface_hub google-genai gradio -q

In [3]:
from huggingface_hub import InferenceClient
from google import genai
from google.genai import types
from google.colab import userdata

# Os tokens ficam nos Secrets do Colab (ícone de chave na barra lateral)
HF_TOKEN = userdata.get("HF_TOKEN")
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

print("HF_TOKEN ok" if HF_TOKEN else "ERRO: adicione HF_TOKEN nos Secrets do Colab")
print("GEMINI_API_KEY ok" if GEMINI_API_KEY else "ERRO: adicione GEMINI_API_KEY nos Secrets do Colab")

HF_TOKEN ok
GEMINI_API_KEY ok


In [11]:
# Modelos usados na atividade (os mesmos da Aula 05)
MODELO_HF = "meta-llama/Llama-3.1-8B-Instruct"
MODELO_GEMINI = "gemini-3.5-flash-lite"

cliente_hf = InferenceClient(model=MODELO_HF, token=HF_TOKEN, provider="auto")
cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)

In [5]:
# >>> PERSONALIZE: descreva o assistente do grupo de acordo com o tema escolhido.
# Este é o "system": a instrução que define o comportamento do assistente.
# O exemplo abaixo é do tema 1 (casa inteligente). Troque pelo tema do grupo.

SYSTEM_PROMPT = """
Você é um especialista em eficiência energética residencial e comportamento de consumo sustentável. Sua missão é analisar dados de consumo de energia elétrica vindos de medidores inteligentes (smart meters) e tomadas conectadas (smart plugs) para fornecer dicas de economia personalizadas, práticas e fáceis de entender.

Para isso, considere os seguintes dados de entrada fictícios (ou fornecidos pelo usuário):
1. Consumo geral da casa (picos de horário do medidor inteligente).
2. Consumo individual por aparelho (dados das tomadas conectadas, ex: geladeira, ar-condicionado, computador).
3. Padrões de rotina (horários em que a casa fica vazia ou com maior atividade).

Com base nessas premissas, gere um relatório estruturado contendo:

1. **Análise de Vilões do Consumo:** Identifique quais aparelhos ou tomadas conectadas estão consumindo mais energia (em kWh ou proporção) e se há consumo residual ("vampiro") em aparelhos que deveriam estar desligados.
2. **Alertas de Horário de Pico:** Indique se o maior consumo está ocorrendo nos horários em que a tarifa de energia é mais cara (se aplicável) ou em momentos de desperdício (ex: luzes/ar ligados na casa vazia).
3. **Plano de Ação Prático (Dicas de Consumo):** Liste de 3 a 5 ações imediatas e fáceis que o usuário pode tomar para reduzir a conta. Use uma linguagem simples e direta.
4. **Sugestões de Automação:** Recomende regras de automação inteligentes usando as próprias tomadas conectadas (ex: "desligar a tomada X automaticamente após as 23h").

Mantenha o tom amigável, motivador e focado em economia financeira e sustentabilidade. Evite termos técnicos complexos sem explicação.

"""

---
## Etapa 1: Assistente com personalidade (Hugging Face)

Cada mensagem enviada ao modelo tem uma etiqueta `role` que diz quem escreveu:

- `system`: a regra que o assistente deve seguir
- `user`: a pergunta do usuário

Nesta etapa, a **mesma pergunta** é enviada três vezes, e **só o `system` muda**:

| Chamada | system | user |
|---|---|---|
| 1 | Especialista no tema do grupo | mesma pergunta |
| 2 | Professor para crianças | mesma pergunta |
| 3 | Resposta em uma frase | mesma pergunta |

Se as respostas saírem diferentes, a diferença veio só do `system`. É assim que se criam assistentes diferentes em cima do mesmo modelo.

In [6]:
def perguntar_hf(pergunta, system, temperatura=0.7):
    """Envia uma pergunta ao modelo do Hugging Face e devolve o texto da resposta."""
    mensagens = [
        {"role": "system", "content": system},   # como o assistente deve se comportar
        {"role": "user",   "content": pergunta}, # o que o usuário perguntou
    ]
    resposta = cliente_hf.chat_completion(
        messages=mensagens,
        max_tokens=300,
        temperature=temperatura,
    )
    return resposta.choices[0].message.content

In [8]:
# >>> PERSONALIZE: crie 3 personalidades diferentes para o assistente do grupo.
personalidades = {
    "Especialista": SYSTEM_PROMPT,
    "Nerd": "Você segue este prompt, mas sempre responde o usuário fazendo múltiplas referências à cultura popular:"+SYSTEM_PROMPT,
    "Poeta": "Você segue este prompt, mas sempre responde o usuário de maneira poética:"+SYSTEM_PROMPT,
}

# >>> PERSONALIZE: uma pergunta relacionada ao tema do grupo.
pergunta = "Como um sensor de presença pode ajudar a economizar energia em casa?"

for nome, system in personalidades.items():
    print(f"===== {nome} =====")
    print(perguntar_hf(pergunta, system))
    print()

===== Especialista =====
Um sensor de presença é uma ferramenta incrível que pode ajudar a economizar energia em casa de maneiras inteligentes e práticas. Aqui estão algumas maneiras pelas quais um sensor de presença pode fazer a diferença:

1. **Automatizar a iluminação:** Com um sensor de presença, você pode programar suas lâmpadas para se ligar apenas quando alguém está em casa, economizando energia e reduzindo o desperdício. Se você costuma sair de casa durante o dia, por exemplo, é provável que as luzes estejam acesas sem uso.

2. **Controle de Aparelhos:** Você pode usar sensores de presença para controlar a utilização de aparelhos como a televisão, computador, ou até mesmo a secadora de cabelo. Se alguém está em casa, esses aparelhos podem se ligar automaticamente. Se a casa estiver vazia, eles podem se desligar.

3. **Monitoramento de Temperatura:** Um sensor de presença pode ajudar a manter a temperatura ideal na sua casa. Se alguém está em casa, você pode programar o sistema 

**Observações do grupo (Etapa 1):**

- O que mudou nas respostas de cada personalidade?
  As personalidades poeta e nerd, apesar de se centrarem no prompt principal enviado, seguiram também as instruções únicas concatenadas ao prompt.

- Qual `system` gerou a resposta mais útil para o tema? Por quê?
O system principal (especialista), dado que ele seguiu somente o prompt de explicação padrão enviado.
(responda aqui)

---
## Etapa 2: Hugging Face x Gemini

Agora as mesmas perguntas vão para **dois modelos diferentes**.

No Gemini, o `system` não vai dentro da lista de mensagens. Ele é passado no parâmetro `system_instruction`.

In [13]:
def perguntar_gemini(pergunta, system):
    """Envia uma pergunta ao Gemini e devolve o texto da resposta."""
    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=pergunta,
        config=types.GenerateContentConfig(
            system_instruction=system,  # equivalente ao role "system"
            max_output_tokens=300,
        ),
    )
    return resposta.text

In [14]:
# >>> PERSONALIZE: 3 perguntas sobre o tema do grupo.
perguntas = [
    "Quais sensores são mais usados em uma casa inteligente?",
    "Qual a diferença entre Wi-Fi e Zigbee para dispositivos da casa?",
    "Quais cuidados de segurança devo ter com câmeras conectadas?",
]

for p in perguntas:
    print("PERGUNTA:", p)
    print("\n--- Hugging Face (Llama) ---")
    print(perguntar_hf(p, SYSTEM_PROMPT))
    print("\n--- Gemini ---")
    print(perguntar_gemini(p, SYSTEM_PROMPT))
    print("\n" + "=" * 60 + "\n")

PERGUNTA: Quais sensores são mais usados em uma casa inteligente?

--- Hugging Face (Llama) ---
Em uma casa inteligente, os sensores mais comuns incluem:

1. **Sensores de presença (PIR - Passive Infrared)**: Detectam a presença de pessoas em uma área, ativando dispositivos como iluminação, ar condicionado ou sistemas de segurança.
2. **Sensores de temperatura e umidade**: Regulam o controle de temperatura, sistemas de ar condicionado, e podem ajudar na automação de iluminação e outros dispositivos.
3. **Sensores de movimento (MAG)**: Monitoram o movimento de objetos, pessoas ou animais em uma área, ativando alarmes, iluminação ou outros dispositivos.
4. **Sensores de pressão**: Medem a pressão do ar em sistemas de ar condicionado, detectando se há problemas no sistema.
5. **Sensores de nível de água**: Verificam o nível de água nos tanques de água, alertando sobre a necessidade de reabastecimento.
6. **Sensores de gás**: Detectam a presença de gases nocivos, como monóxido de carbono, 

**Observações do grupo (Etapa 2):**

- Os dois modelos seguiram as regras do `SYSTEM_PROMPT` (idioma, tamanho, tema)?
Sim. Nenhum deles escapou do tema, tamanho ou idioma.
- Qual respondeu melhor? Em qual pergunta a diferença foi maior?
O gemini manteve o ar de afabilidade, enquanto o modelo do huggingface foi mais direto. Portanto o gemini seguiu mais apropriadamente o contexto do prompt enviado.

(responda aqui)

---
## Etapa 3: Chat com memória

O modelo **não guarda memória** entre uma chamada e outra.
Quem guarda a conversa é o nosso código, na lista `historico`, que é enviada inteira a cada mensagem.

Comandos do chat:
- `sair` encerra o chat
- `limpar` apaga o histórico (o assistente "esquece" a conversa)
- `historico` mostra quantas mensagens estão guardadas

**Teste sugerido:** diga seu nome, pergunte "qual é o meu nome?", digite `limpar` e pergunte de novo.

In [16]:
historico = [{"role": "system", "content": SYSTEM_PROMPT}]

print("Chat iniciado! Comandos: sair | limpar | historico\n")

while True:
    entrada = input("Você: ")

    if entrada.lower() == "sair":
        print("Encerrando chat.")
        break

    if entrada.lower() == "limpar":
        historico = [{"role": "system", "content": SYSTEM_PROMPT}]  # mantém só o system
        print("\n(histórico apagado)\n")
        continue

    if entrada.lower() == "historico":
        print(f"\n(mensagens no histórico: {len(historico)})\n")
        continue

    historico.append({"role": "user", "content": entrada})

    resposta = cliente_hf.chat_completion(messages=historico, max_tokens=300)
    texto = resposta.choices[0].message.content

    historico.append({"role": "assistant", "content": texto})

    print(f"\nAssistente: {texto}\n")

Chat iniciado! Comandos: sair | limpar | historico

Você: Olá, meu nome é Soteras, qual é seu nome?

Assistente: Olá Soteras! Meu nome é Energi, mas eu sou um especialista em eficiência energética e comportamento de consumo sustentável. Estou aqui para ajudar você a economizar energia e dinheiro!

Parece que você tem dados de consumo de energia elétrica vindos de medidores inteligentes e tomadas conectadas. Estou ansioso para analisar esses dados e fornecer dicas práticas e fáceis de entender para que você possa reduzir sua conta de energia!

Por favor, me forneça os dados que você tem e vamos começar a trabalhar em conjunto para criar um plano de ação prático e eficaz!

Você: limpar

(histórico apagado)

Você: olá, voce sabe qual é meu nome?

Assistente: Peço desculpas, mas não tenho conhecimento sobre sua identidade pessoal. Estou aqui para ajudar com questões sobre eficiência energética e consumo sustentável, mas não tenho acesso a informações sobre você como pessoa. Se você quiser 

KeyboardInterrupt: Interrupted by user

In [ ]:
# Veja como ficou a lista enviada ao modelo
for msg in historico:
    print(f"[{msg['role']}] {msg['content'][:80].replace(chr(10), ' ')}")

**Observações do grupo (Etapa 3):**

- O que aconteceu quando vocês perguntaram o nome antes e depois do `limpar`?
O contexto da ia foi reiniciado e, portanto, sua memória de meu nome foi perdida.
- Explique, com suas palavras, por que isso acontece.
Pois a IA não possui memória no termo convencional da palavra, o que ela tem é um registro de contexto que tem que ser enviado para a parte generativa para que a resposta possa levar em consideração os eventos que aconteceram anteriormente.
(responda aqui)

---
## Etapa 4: Interface web com Gradio

O assistente ganha uma interface web, com a opção de escolher o modelo (Hugging Face ou Gemini).

O Gradio entrega o histórico da conversa já no formato de `role`/`content`.
A função `texto_da_mensagem` existe porque, dependendo da versão do Gradio, o conteúdo chega como texto simples ou como lista.

In [17]:
import gradio as gr

def texto_da_mensagem(conteudo):
    """Extrai o texto de uma mensagem do histórico do Gradio."""
    if isinstance(conteudo, str):
        return conteudo
    if isinstance(conteudo, list):
        return " ".join(item.get("text", "") for item in conteudo if isinstance(item, dict))
    return str(conteudo)


def responder(mensagem, historico_gradio, provedor):
    if provedor == "Hugging Face":
        # Formato HF: roles system, user e assistant
        mensagens = [{"role": "system", "content": SYSTEM_PROMPT}]
        for msg in historico_gradio:
            mensagens.append({"role": msg["role"], "content": texto_da_mensagem(msg["content"])})
        mensagens.append({"role": "user", "content": mensagem})

        resposta = cliente_hf.chat_completion(messages=mensagens, max_tokens=300)
        return resposta.choices[0].message.content

    else:
        # Formato Gemini: roles user e model; o system vai em system_instruction
        conteudos = []
        for msg in historico_gradio:
            papel = "model" if msg["role"] == "assistant" else "user"
            conteudos.append({"role": papel, "parts": [{"text": texto_da_mensagem(msg["content"])}]})
        conteudos.append({"role": "user", "parts": [{"text": mensagem}]})

        resposta = cliente_gemini.models.generate_content(
            model=MODELO_GEMINI,
            contents=conteudos,
            config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, max_output_tokens=300),
        )
        return resposta.text

In [18]:
# >>> PERSONALIZE: título, descrição e exemplos de acordo com o tema do grupo.
gr.ChatInterface(
    fn=responder,
    additional_inputs=[gr.Radio(["Hugging Face", "Gemini"], value="Hugging Face", label="Modelo")],
    title="Assistente de Economia Energética",
    description="Pergunte sobre técnicas de economia de energia, tecnologia verde e relacionados.",
    examples=[
        ["Como gastar menos energia?", "Hugging Face"],
        ["Vale a pena usar tomadas conectadas?", "Gemini"],
    ],
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://34ef34b3d304b4d734.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**Observações do grupo (Etapa 4):**

- Tire um print da interface funcionando e coloque no README do repositório do grupo.
- Troque de modelo no meio da conversa. O assistente continuou lembrando do que foi dito? Por quê?
Eu perguntei o que eu tinha acabado de perguntar para ele, levando em consideração que a última pergunta tinha sido para o outro modelo e ele conseguiu se lembrar do contexto, isso se deve pois o histórico da conversa é enviado na requisição para manter a memória do modelo.

(responda aqui)

> Para parar a interface, interrompa a célula (botão de parar do Colab).

---
## Etapa 5 (Bônus): API com FastAPI

Aqui o assistente vira uma **API REST**, como no final da Aula 05.
Qualquer frontend (site, app, dispositivo IoT) poderia chamar esse endpoint.

A célula abaixo cria o arquivo `app.py`.

In [19]:
%%writefile app.py
import os
from fastapi import FastAPI
from pydantic import BaseModel
from huggingface_hub import InferenceClient

# O token e o system prompt vêm de variáveis de ambiente (nunca escreva o token no código)
client = InferenceClient(
    model="meta-llama/Llama-3.1-8B-Instruct",
    token=os.environ["HF_TOKEN"],
    provider="auto",
)
SYSTEM_PROMPT = os.environ.get("SYSTEM_PROMPT", "Você é um assistente prestativo.")

app = FastAPI()

class Pergunta(BaseModel):
    mensagem: str

@app.post("/chat")
def chat(pergunta: Pergunta):
    resposta = client.chat_completion(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": pergunta.mensagem},
        ],
        max_tokens=300,
    )
    return {"resposta": resposta.choices[0].message.content}

Writing app.py


In [20]:
# Inicia o servidor em segundo plano, dentro do próprio Colab
!pip install fastapi uvicorn -q

import os, subprocess, time
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["SYSTEM_PROMPT"] = SYSTEM_PROMPT

servidor = subprocess.Popen(["uvicorn", "app:app", "--port", "8000"])
time.sleep(5)  # espera o servidor subir
print("Servidor rodando em http://localhost:8000")

Servidor rodando em http://localhost:8000


In [21]:
# Testa o endpoint como um frontend faria (HTTP POST com JSON)
import requests

r = requests.post("http://localhost:8000/chat", json={"mensagem": "O que é IoT?"})
print(r.status_code)
print(r.json()["resposta"])

200
O Internet das Coisas (IoT) é uma tecnologia que permite a conexão e comunicação entre dispositivos e objetos, como aparelhos eletrônicos, sensores e atuadores, por meio da internet. Isso permite que esses dispositivos coletem e compartilhem dados em tempo real, e que possam ser controlados e monitorados remotamente.

No contexto do seu pedido, o IoT é aplicado em dispositivos como medidores inteligentes (smart meters) e tomadas conectadas (smart plugs), que podem coletar dados sobre o consumo de energia e fornecer informações precisas sobre o uso de energia na casa. Isso permite que você, como especialista em eficiência energética residencial e comportamento de consumo sustentável, forneça dicas personalizadas para reduzir o consumo de energia e economizar dinheiro.

Alguns exemplos de como o IoT pode ser aplicado na sua missão incluem:

* Automatizar o desligamento de dispositivos quando não estão em uso, como aparelhos de TV ou computadores.
* Receber alertas quando um aparelho 

In [22]:
# Encerra o servidor
servidor.terminate()
print("Servidor encerrado.")

Servidor encerrado.
